# OAR Full Traceability Table

**Goal:** Build a human-readable table that shows the full chain  
**human phrases → extracted O/A/R → canonical synset / verb**  
for every image in the human (B) condition.

Two tables are produced:

1. **`oar_image_traceability.csv`** — one row per image. Compact view with phrases and `raw → canonical` strings for every O/A/R. Use this for paper appendices or supplementary slides.
2. **`oar_relationships_long.csv`** — one row per relationship triple. Best for sorting/filtering by predicate or subject/object synset.

**Note:** OAR extraction in this pipeline is *image-level* — phrases are pooled per image, so individual O/A/R facts can be traced to the image's full phrase set but not to a single phrase.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, 'scripts')
from _vc_canon import resolve_predicate, normalize_attr
from _vc_canon_extended_B import resolve_object_synset_extended as resolve_object_synset

ROOT      = Path('.')
THREE_DIR = ROOT / 'vc_genome_output_full' / 'three_conditions'
OUT       = ROOT / 'vc_genome_output_full' / 'vistype_profile'
OUT.mkdir(parents=True, exist_ok=True)

with open(THREE_DIR / 'oar_B_510.json', encoding='utf-8') as f:
    b_raw = json.load(f)

phrases_df = pd.read_csv(
    'phrase_reduction_v2/image_compiled_phrases.csv',
    usecols=['imageName', 'VisType', 'rawUserComments',
             'humanCuratedPhrases', 'finalPhrases']
).drop_duplicates('imageName').set_index('imageName')

print(f'Loaded {len(b_raw)} images with OAR extractions')
print(f'Phrase metadata available for {len(phrases_df)} images')
print(f'Intersection: {len(set(b_raw) & set(phrases_df.index))} images')


Loaded 510 images with OAR extractions
Phrase metadata available for 520 images
Intersection: 510 images


## 1. Image-level traceability table (wide format)
One row per image. Each O/A/R group is collapsed into a single readable string.

In [14]:
def fmt_object(obj, syn):
    """e.g. 'bar --> mark.bar' or 'bar --> mark.bar [legend]' if region given."""
    region = obj.get('region', '')
    base = f"{obj['name']} --> {syn}"
    return f"{base} [{region}]" if region else base

def fmt_attr(attr_row, obj_lookup):
    """e.g. 'hard_to_read @ bar (mark.bar) [+]'"""
    raw  = attr_row.get('attr', attr_row.get('name', ''))
    norm = normalize_attr(raw)
    oid  = attr_row.get('object_id')
    obj_raw, obj_syn = obj_lookup.get(oid, ('?', '?'))
    sent = attr_row.get('sentiment', '')
    body = f"{norm} @ {obj_raw} ({obj_syn})"
    return f"{body} [{sent}]" if sent else body

def fmt_rel(rel, obj_lookup):
    """e.g. 'bar (mark.bar) --hinders_reading[raw: increases_difficulty_of]--> label (text.label) [+]'"""
    s_raw, s_syn = obj_lookup.get(rel.get('subj'), ('?', '?'))
    o_raw, o_syn = obj_lookup.get(rel.get('obj'),  ('?', '?'))
    p_raw = rel.get('pred', '')
    p_can = resolve_predicate(p_raw)
    sent  = rel.get('sentiment', '')
    pred_str = p_can if p_can == p_raw else f"{p_can}(raw:{p_raw})"
    body = f"{s_raw}({s_syn}) --{pred_str}--> {o_raw}({o_syn})"
    return f"{body} [{sent}]" if sent else body

rows = []
for img, ext in b_raw.items():
    meta = phrases_df.loc[img] if img in phrases_df.index else None
    objs  = ext.get('objects', [])
    attrs = ext.get('attributes', [])
    rels  = ext.get('relationships', [])
    # Build obj_id -> (raw, synset) lookup for this image
    obj_lookup = {o['id']: (o['name'], resolve_object_synset(o['name'])) for o in objs}
    rows.append({
        'imageName':           img,
        'VisType':             meta['VisType'] if meta is not None else '',
        'rawUserComments':     (meta['rawUserComments'] if meta is not None else ''),
        'humanCuratedPhrases': (meta['humanCuratedPhrases'] if meta is not None else ''),
        'finalPhrases':        (meta['finalPhrases'] if meta is not None else ''),
        'n_obj':  len(objs),
        'n_attr': len(attrs),
        'n_rel':  len(rels),
        'objects':       ' | '.join(fmt_object(o, obj_lookup[o['id']][1]) for o in objs),
        'attributes':    ' | '.join(fmt_attr(a, obj_lookup) for a in attrs),
        'relationships': ' | '.join(fmt_rel(r, obj_lookup) for r in rels),
    })

image_df = pd.DataFrame(rows)
image_df.to_csv(OUT / 'oar_image_traceability.csv', index=False)
print(f'Saved {len(image_df)} rows \u2192 oar_image_traceability.csv')
display(image_df.head(3))

Saved 510 rows → oar_image_traceability.csv


,imageName,VisType,rawUserComments,humanCuratedPhrases,finalPhrases,n_obj,n_attr,n_rel,objects,attributes,relationships
0,whoO06_2.png,Bar,has a title and legend,title/axis/label/descriptions; too much legend,labels/axes/legends,2,2,1,legend --> furniture.legend [legend] | title -...,increases_perceived_complexity @ legend (furni...,legend(furniture.legend) --adds_complexity_rel...
1,visMost97.png,Area,"uses common knowledge, is easy to understand, ...",simple information; familiar representation; c...,color variety/shading; domain-specific concept...,3,4,2,chart --> whole.visualization [overall] | colo...,simple_information @ data_content (content.dat...,color_encoding(property.color) --aids_interpre...
2,wsj135.png,Bar,Left one is just a time scale with 2 data sets...,much/more data/info/info spread; scale differe...,color variety/shading; easy/hard to interpret;...,5,4,2,chart --> whole.visualization [overall] | time...,time_scale_present @ time_axis (furniture.axes...,dataset_1(content.data) --distinguished_from--...


## 2. Relationship triples (long format)
One row per relationship — easy to sort by predicate or any synset column.

In [16]:
rel_rows = []
for img, ext in b_raw.items():
    meta = phrases_df.loc[img] if img in phrases_df.index else None
    obj_lookup = {o['id']: (o['name'], resolve_object_synset(o['name']))
                  for o in ext.get('objects', [])}
    # Attach attribute lists per object id
    attrs_by_obj = {}
    for a in ext.get('attributes', []):
        attrs_by_obj.setdefault(a.get('object_id'), []).append(
            normalize_attr(a.get('attr', a.get('name', '')))
        )
    for r in ext.get('relationships', []):
        s_raw, s_syn = obj_lookup.get(r.get('subj'), ('?', 'unknown.?'))
        o_raw, o_syn = obj_lookup.get(r.get('obj'),  ('?', 'unknown.?'))
        p_raw = r.get('pred', '')
        p_can = resolve_predicate(p_raw)
        rel_rows.append({
            'imageName':     img,
            'VisType':       meta['VisType'] if meta is not None else '',
            'finalPhrases':  (meta['finalPhrases'] if meta is not None else ''),
            'subj_raw':      s_raw,
            'subj_synset':   s_syn,
            'subj_category': s_syn.split('.')[0] if '.' in s_syn else s_syn,
            'subj_attrs':    '; '.join(attrs_by_obj.get(r.get('subj'), [])),
            'pred_raw':      p_raw,
            'pred_canon':    p_can,
            'obj_raw':       o_raw,
            'obj_synset':    o_syn,
            'obj_category':  o_syn.split('.')[0] if '.' in o_syn else o_syn,
            'obj_attrs':     '; '.join(attrs_by_obj.get(r.get('obj'), [])),
            'sentiment':     f"[{r['sentiment']}]" if r.get('sentiment') else '',
            'topic':         r.get('topic', ''),
        })

rel_df = pd.DataFrame(rel_rows)
rel_df.to_csv(OUT / 'oar_relationships_long.csv', index=False)
print(f'Saved {len(rel_df)} relationship triples \u2192 oar_relationships_long.csv')
display(rel_df.head(10))

Saved 916 relationship triples → oar_relationships_long.csv


,imageName,VisType,finalPhrases,subj_raw,subj_synset,subj_category,subj_attrs,pred_raw,pred_canon,obj_raw,obj_synset,obj_category,obj_attrs,sentiment,topic
0,whoO06_2.png,Bar,labels/axes/legends,legend,furniture.legend,furniture,increases_perceived_complexity,adds_complexity_relative_to,adds_complexity_relative_to,title,text.title,text,aids_quick_interpretation,[+],Immediacy / Cognitive Load
1,visMost97.png,Area,color variety/shading; domain-specific concept...,color_encoding,property.color,property,distinguished_distinctive_colors,aids_interpretation_of,aids_interpretation,data_content,content.data,content,simple_information; common_knowledge_domain,[-],Visual Encoding Clarity
2,visMost97.png,Area,color variety/shading; domain-specific concept...,data_content,content.data,content,simple_information; common_knowledge_domain,requires_minimal_cognitive_load,requires_minimal_cognitive_load,chart,whole.visualization,whole,easy_to_understand,[-],Immediacy / Cognitive Load
3,wsj135.png,Bar,color variety/shading; easy/hard to interpret;...,dataset_1,content.data,content,,distinguished_from,distinguished_from,dataset_2,content.data,content,,[-],"Color, Symbol, and Texture Details"
4,wsj135.png,Bar,color variety/shading; easy/hard to interpret;...,dataset_1,content.data,content,,shares_time_axis_with,shares_time_axis_with,dataset_2,content.data,content,,[-],Data Density / Image Clutter
5,InfoVisC.133.5(3).png,Node-link,color contrast/clarity; much/little data/info;...,shapes,mark.shape,mark,varied_shapes_present; high_contrast_marks,increases_visual_complexity_of,increases_complexity,chart,whole.visualization,whole,low_detail_count,[+],Aesthetics Uncertainty
6,InfoVisC.133.5(3).png,Node-link,color contrast/clarity; much/little data/info;...,data_elements,content.data,content,large_element_size,reduces_clutter_in,reduces_clutter_in,chart,whole.visualization,whole,low_detail_count,[-],Data Density / Image Clutter
7,SciVisJ.1025.11.png,Point,amount of words/context/numbers; color contras...,text_labels,text.label,text,lots_of_inscriptions; detailed_annotations; sp...,increases_cognitive_load,increases_effort,data_elements,content.data,content,high_information_volume; nuanced_data_content;...,[+],Immediacy / Cognitive Load
8,SciVisJ.1025.11.png,Point,amount of words/context/numbers; color contras...,numbers,content.number,content,numeric_labels_present,contributes_to_clutter_in,contributes_to_clutter_in,measurements,content.number,content,several_measurements_shown,[+],Data Density / Image Clutter
9,SciVisJ.1025.11.png,Point,amount of words/context/numbers; color contras...,colors,property.color,property,multiple_colors_used; high_color_contrast,aids_differentiation_of,aids_differentiation_of,data_elements,content.data,content,high_information_volume; nuanced_data_content;...,[-],Visual Encoding Clarity


## 3. Sanity-check: pick a random image and print the full chain

In [9]:
import textwrap

sample = image_df.sample(1, random_state=42).iloc[0]
print(f"=== {sample['imageName']}  ({sample['VisType']}) ===\n")
print("PHRASES (humanCuratedPhrases):")
print(textwrap.fill(str(sample['humanCuratedPhrases']), width=100, initial_indent='  ', subsequent_indent='  '))
print(f"\nOBJECTS ({sample['n_obj']}):")
for o in sample['objects'].split(' | '):
    print(f"  {o}")
print(f"\nATTRIBUTES ({sample['n_attr']}):")
for a in sample['attributes'].split(' | '):
    print(f"  {a}")
print(f"\nRELATIONSHIPS ({sample['n_rel']}):")
for r in sample['relationships'].split(' | '):
    print(f"  {r}")

=== MorphableWordClouds8.png  (Text) ===

PHRASES (humanCuratedPhrases):
  too much data/info; understand/read shapes; amount of words/context/numbers; inconsistent; hard to
  read shape

OBJECTS (3):
  word_cloud --> whole.visualization [overall]
  text_labels --> text.label [data_area]
  shape_outline --> mark.shape [data_area]

ATTRIBUTES (5):
  excessive_information_volume @ word_cloud (whole.visualization) [+]
  high_density_writing @ text_labels (text.label) [+]
  inconsistent_text_arrangement @ text_labels (text.label) [+]
  hard_to_read_shape @ shape_outline (mark.shape) [+]
  high_cognitive_load @ word_cloud (whole.visualization) [+]

RELATIONSHIPS (3):
  text_labels(text.label) --clutters--> shape_outline(mark.shape) [+]
  shape_outline(mark.shape) --obscured_by--> text_labels(text.label) [+]
  text_labels(text.label) --lacks_consistency_with--> shape_outline(mark.shape) [+]


## 4. Corpus-level summaries

In [10]:
print('Predicate canonicalisation effect:')
n_raw_pred  = rel_df['pred_raw'].nunique()
n_canon     = rel_df['pred_canon'].nunique()
print(f'  Distinct raw predicates:       {n_raw_pred}')
print(f'  Distinct canonical predicates: {n_canon}')
print(f'  Collapse ratio:                {n_canon/n_raw_pred:.1%}')

print('\nTop 15 canonical predicates by frequency:')
display(rel_df['pred_canon'].value_counts().head(15).to_frame('count'))

print('\nTop 10 subj_synset \u2192 obj_synset patterns:')
display(
    rel_df.groupby(['subj_synset', 'pred_canon', 'obj_synset']).size()
          .sort_values(ascending=False).head(10).to_frame('count')
)

Predicate canonicalisation effect:
  Distinct raw predicates:       451
  Distinct canonical predicates: 422
  Collapse ratio:                93.6%

Top 15 canonical predicates by frequency:


,count
pred_canon,
increases_effort,69
increases_clutter_in,40
increases_complexity,34
contributes_to,33
aids_interpretation,21
contributes_to_clutter_in,21
combined_with,15
encoded_with,15
differentiates,11



Top 10 subj_synset → obj_synset patterns:


count
subj_synset      pred_canon           obj_synset                
content.data     increases_effort     whole.visualization     14
property.color   increases_complexity whole.visualization     14
                 increases_effort     whole.visualization      7
mark.shape       encoded_with         property.color           6
mark.point       increases_clutter_in whole.visualization      6
property.color   increases_clutter_in whole.visualization      6
text.label       increases_effort     whole.visualization      6
mark.line        increases_clutter_in whole.visualization      5
property.color   contributes_to       whole.visualization      5
structure.layout increases_effort     whole.visualization      5